![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 44 -- Lab 1: Every Pixel Counts

A photo app wants to know what every pixel of a photo shows: a person, a car, a dog, a bicycle, a chair, or none of these. You write the program that paints every pixel of a new photo with its class. Your file is scored with mIoU: for each of the five classes, the pixels you and the truth both give that class, divided by the pixels either of you gives it; then the mean of the five. Higher is better.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** Everyday photos, each resized to 256 x 256 pixels, each holding at least one of the five classes. The training photos come with their masks; the test photos do not, and only the scorer holds their masks. The setup cell leaves in `DATA_DIR`: `train.csv` (`id`, `image`, `mask`: the paths of a photo and its mask), `test.csv` (`id`, `image`), `sample_submission.csv` (the format of the file you hand in) and the folders `train_images/`, `train_masks/` and `test_images/`. **Use a GPU:** in Colab, Runtime, Change runtime type, T4 GPU, before you run the first cell.

Your work is checked as you go, and the clock started at the opening time, so start at once and run every checkpoint cell.

---
# Setup

Run the four cells below in order. The first one installs the checkpoint library where it is missing. The second one opens your session with the scorer: replace `PASTE-YOUR-ID` by your own student id before you run it. From that cell on, this notebook records cell runs, cell source and errors for assessment. The third one imports the libraries and the fourth one gets the data.

Your id is yours alone. Do not share a copy of this notebook that holds it.

In [ ]:
# Installs the checkpoint library where it is missing. On Colab that takes a few seconds.
import importlib.util

if importlib.util.find_spec("olympiad") is None:
    !pip -q install olympiad-lab
    importlib.invalidate_caches()


In [ ]:
import os
from olympiad import ck

STUDENT_ID = os.environ.get("OLYMPIAD_STUDENT_ID", "PASTE-YOUR-ID")
ck.start(lab="cv-01", student_id=STUDENT_ID)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

print("libraries ready")


In [ ]:
import os

DATA_DIR = os.environ.get("OLYMPIAD_DATA_DIR")      # set when the instructor runs the notebook again
if not DATA_DIR:
    import kagglehub
    DATA_DIR = kagglehub.dataset_download("sattamjaltwaim/every-pixel-counts")
print("Data folder:", DATA_DIR)
print(sorted(os.listdir(DATA_DIR))[:20])


---
# The task

**The story.** A photo app wants to understand photos pixel by pixel. For every pixel it must say what the pixel
shows: a person, a car, a dog, a bicycle, a chair, or none of these. This is called **semantic segmentation**: a
class for every pixel, with no difference between two people (both are "person").

**One data point** is one photo and its mask:

- the **photo**, `train_images/<id>.jpg`: an array of shape (256, 256, 3), red, green and blue values from 0 to 255;
- the **mask**, `train_masks/<id>.png`: an array of shape (256, 256), one class number per pixel.

**The answer** is the mask of a test photo. Its class numbers:

| 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| background (anything else) | person | car | dog | bicycle | chair |

For example, a photo of a girl on a sofa with her dog has 1 on the girl's pixels, 3 on the dog's and 0 everywhere
else, the sofa included: a sofa is none of the five. Every photo holds at least one of the five classes. Opened as a
picture, a mask looks almost black: 5 is very dark on a scale that goes to 255. Draw it with a colour scale.

**The files.** The first rows of `train.csv` and `test.csv`:

| id | image | mask |
|---|---|---|
| 0 | train_images/0.jpg | train_masks/0.png |
| 1 | train_images/1.jpg | train_masks/1.png |

| id | image |
|---|---|
| 726 | test_images/726.jpg |
| 727 | test_images/727.jpg |

**The organisers' model.** *"A pretrained ResNet34 model is used as the encoder of the U-Net."* In plain words: the
model is a U-Net, and its encoder (the half that goes down) is a ResNet-34 whose weights were already trained on
millions of photos. Today you use it in two stages: first with those weights frozen, then with them trained further.

**The file you hand in**, `submission.csv`, has one row per test photo: its `id` and its `mask` written as text. The
mask is written as five **run-length masks**, one per class (person first, chair last), joined by ` | `. A run-length
mask lists pairs `start length`: number the 65,536 pixels from 1, row by row; the pair `3 4` marks pixels 3, 4, 5
and 6. A class with no pixel is an empty part. Two rows of a valid file (the numbers are made up):

| id | mask |
|---|---|
| 726 | `30001 12 30257 14 \| \| \| \|` |
| 727 | ` \| \| 2 24 258 25 \| \|` |

The first row says: person on pixels 30,001 to 30,012 and 30,257 to 30,270, no other class. The second: a dog on
pixels 2 to 25 and 258 to 282. `sample_submission.csv` has every photo all background. The starter's `rle_encode`
writes one run-length mask from a 0/1 mask; use it.

**The score** is mIoU on the test photos, computed by the scorer cell in Part 2. Higher is better; 1 would be
perfect.

**The rules.** Any library; no outside data (the pretrained ResNet-34 weights are allowed). Your notebook must run
from start to end.

**The starter** (next cell) trains the organisers' small example network for five epochs. The score of its file is
the **baseline**; your points count from there.

**Today you work alone.** Nobody helps with the code and AI assistants are off. This morning's slides are yours
to use.


---
# The starter solution

Every olympiad task comes with a starter: something that hands in a valid file. Its score is the **baseline**, and your points count up from it. This task came with a runnable example: a small network of two convolutions down and two up, trained for five epochs. Below it is adapted to these files and made to write a valid file (the example wrote none). Read it before you begin; you may reuse any of its lines, the file writing above all.

```python
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
torch.manual_seed(0)
MEAN = np.array([0.485, 0.456, 0.406])
STD = np.array([0.229, 0.224, 0.225])


class Photos(Dataset):
    def __init__(self, table, with_masks):
        self.table = table
        self.with_masks = with_masks

    def __len__(self):
        return len(self.table)

    def __getitem__(self, i):
        image = np.array(Image.open(os.path.join(DATA_DIR, self.table["image"][i])))
        x = torch.tensor(((image / 255 - MEAN) / STD).transpose(2, 0, 1), dtype=torch.float32)
        if not self.with_masks:
            return x
        mask = np.array(Image.open(os.path.join(DATA_DIR, self.table["mask"][i])))
        return x, torch.tensor(mask, dtype=torch.long)


class SimpleSegmentationModel(nn.Module):
    def __init__(self, num_classes=6):
        super(SimpleSegmentationModel, self).__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        # Decoder
        self.decoder = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, num_classes, kernel_size=3, padding=1),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x


model = SimpleSegmentationModel().to(device)
criterion = nn.CrossEntropyLoss(ignore_index=255)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Training: five epochs over every training photo
train_loader = DataLoader(Photos(train, True), batch_size=8, shuffle=True)
model.train()
for epoch in range(5):
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(inputs), labels)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch + 1}, last loss {loss.item():.4f}")


def rle_encode(mask):
    """Run-length encoding of a 0/1 mask: pairs "start length", pixels counted from 1, row by row."""
    pixels = np.concatenate([[0], mask.flatten().astype(np.int8), [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)


# The class map of every test photo, written as one run-length mask per class (1 to 5), joined by " | "
model.eval()
rows = []
with torch.no_grad():
    for inputs in DataLoader(Photos(test, False), batch_size=32):
        class_maps = model(inputs.to(device)).argmax(dim=1).cpu().numpy()
        for class_map in class_maps:
            parts = []
            for c in range(1, 6):
                parts.append(rle_encode(class_map == c))
            rows.append(" | ".join(parts))

submission = pd.DataFrame({"id": test["id"], "mask": rows})
submission.to_csv("submission.csv", index=False)
print(f"Submission file saved: {len(submission)} rows")
```

---
# Part 1 -- Load and look

Every task begins the same way: get the files into memory, then look at the data before you build anything.

## Task 1: Load the files

Read the three CSV files of `DATA_DIR` into `train`, `test` and `sample`. Then read every photo and mask they list into three arrays: `train_images` (one photo per row), `train_masks` (their masks, in the same order) and `test_images`. Print the shapes of the three arrays.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.load(train=train, test=test, sample=sample, train_images=train_images, train_masks=train_masks, test_images=test_images)


## Task 2: Look at a photo and its mask

Store the first four training photos in `first_images` and the number of training photos in `n_images`. Draw photo 0 beside its mask, with a colour bar beside the mask, and print the class numbers that appear in that mask.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.look(first_images, size=n_images)


---
# Part 2 -- Read the task

Before any model comes the task reading: what the metric is, what two answers without a model would score, what the metric rewards, which format the file must have and what you noticed about the task. It is sent by the cell after Task 7. It earns full credit if it arrives by minute 25; after that it earns less, down to nothing at the end of the lab. It is scored after the lab, so no tick appears.

The scorer of this task. Run the cell; you will use `score` all afternoon. It takes two lists (or arrays) of masks.

In [ ]:
def score(true_masks, pred_masks):
    """mIoU, the score of this task. Masks hold a class number per pixel (0 = background, not scored).
    For each class 1 to 5: pixels both masks give it / pixels either mask gives it, over all the photos
    together; then the mean over the classes that appear. Higher is better; 1 is perfect."""
    ious = []
    for c in range(1, 6):
        both = 0
        either = 0
        for true_mask, pred_mask in zip(true_masks, pred_masks):
            both = both + np.logical_and(true_mask == c, pred_mask == c).sum()
            either = either + np.logical_or(true_mask == c, pred_mask == c).sum()
        if either > 0:
            ious.append(both / either)
    return float(np.mean(ious))


true_example = np.array([[1, 1, 0], [0, 2, 2]])
pred_example = np.array([[1, 0, 0], [0, 2, 2]])
print(round(score([true_example], [pred_example]), 3))


## Task 3: What the scorer gives

The scorer cell above prints 0.75 for one tiny mask of 2 x 3 pixels. Which sentence says what `score` gives?

- **a** The share of all pixels whose class is right, background included: 5 of the 6 pixels are right here, and the background pixels count for less.
- **b** For each class that appears, the pixels both masks give it divided by the pixels either mask gives it (class 1: 1 / 2, class 2: 2 / 2), then the mean: (0.5 + 1.0) / 2 = 0.75.
- **c** For each class that appears, the pixels both masks give it divided by the pixels the true mask gives it (class 1: 1 / 2, class 2: 2 / 2), then the mean: (0.5 + 1.0) / 2 = 0.75.

Store the letter in `formula`.

In [ ]:
# Your code here


## Task 4: Two answers without a model

What would two files that need no model score? `background` gives every pixel of every photo the class 0; `person` gives every pixel the class 1. Score both on the masks of `train_masks` with `score`, and store the two numbers in the dictionary `trivial` under these two names.

In [ ]:
# Your code here


## Task 5: What the metric rewards

File A finds every person, car and dog perfectly, but never marks a bicycle or a chair. File B gets an IoU of 0.7 on each of the five classes. Which file has the higher score?

- **a** File A: it is perfect on three classes, and bicycles and chairs cover only a few pixels of the photos.
- **b** File B: each class is one fifth of the score, so A gets (1 + 1 + 1 + 0 + 0) / 5 = 0.6 and B gets 0.7.
- **c** Neither: both files get most of the pixels of the photos right, so they get the same score.

Store the letter in `rewards`.

In [ ]:
# Your code here


## Task 6: The format of the file

Store the format rules in the dictionary `format_rules`: `n_rows`, the number of rows your file must have, `id_col`, the name of its id column, and `pred_col`, the name of the column of your masks. Read all three from what you loaded; do not type them.

In [ ]:
# Your code here


## Task 7: What you noticed

Write in `facts`, in one to three sentences of your own, what you noticed about this task that its statement does not say and that could change your score.

In [ ]:
# Your code here


Run the cell to send your task reading. It answers with one line and shows no score.

In [ ]:
ck.reading(formula=formula, trivial=trivial, rewards=rewards, format=format_rules, facts=facts)


---
# Part 3 -- A first file: the frozen encoder

The simplest pipeline that ends in a file comes first: aim to hand it in within the first hour. Stage 1 keeps the pretrained encoder as it is; only the rest of the model learns.

## Task 8: Hold out validation photos

Split the ids of `train` into `train_ids` and `val_ids`, one photo in five for validation, with `random_state=0`. Print the number of photos in each part.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.split(train_ids=train_ids, val_ids=val_ids)


## Task 9: Batches of photos and masks

Build `train_loader`, the loader of batches your model learns from: the photos of `train_ids` as tensors of shape (3, 256, 256) scaled as the pretrained encoder expects, their masks as tensors of whole numbers of shape (256, 256), 16 photos per batch, shuffled. Keep its first batch in `xb` (photos) and `yb` (masks), and print their shapes and types.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.batch(xb, yb)


## Task 10: The U-Net with its pretrained encoder, frozen

Build `model`: a U-Net whose encoder is a ResNet-34 with its pretrained weights, giving 6 outputs per pixel (one per class, background included), on the GPU. Freeze the encoder: its weights must not change in this stage. Print the shape of what `model` gives for `xb`.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.model(model, xb, stage="frozen")


## Task 11: Train the rest of the model

Train `model` on `train_loader` for a few epochs with a loss for classes, the loop written out. Keep the loss of every step, in order, in the list `losses`.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.loop(losses, stage="frozen")


## Task 12: Score the validation photos

Predict the class map of every photo of `val_ids` (the class with the highest output at each pixel), and store in `frozen_score` their `score` against their true masks: one number. Print it.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.eval(frozen_score, metric="miou_rle", stage="frozen")


## Task 13: Write your first file

Predict the class map of every test photo and write `submission_frozen.csv` in the format of `sample`: for every test photo, its `id` and its mask as text. Leave the index out. Print its shape.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.submit("submission_frozen.csv", stage="frozen")


---
# Part 4 -- Debugging drill

A classmate wanted to look at the first photo of your batch. The cell stops with an error. Run the first cell: it starts the drill's clock. Then repair the classmate's cell until it draws the photo, and run the check. The clock stops at the first check that passes. Read the error message before you change anything.

In [ ]:
ck.drill_start("bug-shp-1")


## Task 14: Repair the classmate's cell

Change as little as you can: `drill_image` must still be the first photo of `xb`, ready to draw.

In [ ]:
drill_image = xb[0].numpy()
plt.imshow(drill_image)
plt.title("the first photo of the batch, as the model reads it")
plt.show()


In [ ]:
ck.drill_check("bug-shp-1", drill_image)


---
# Part 5 -- Stage 2: fine-tuning

The same model again: now its pretrained encoder learns too, with a smaller learning rate, so that its weights move only a little.

## Task 15: Unfreeze the encoder

Let every weight of `model` learn again, and make a new optimizer with a learning rate ten times smaller than in stage 1. Print how many parameters learn now.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.model(model, xb, stage="finetuned")


## Task 16: Fine-tune

Train `model` further on `train_loader`, the loop written out again. Keep the loss of every step, in order, in the new list `finetune_losses`.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.loop(finetune_losses, stage="finetuned")


## Task 17: Score the validation photos again

Store in `finetuned_score` the `score` of the fine-tuned model's class maps of the photos of `val_ids` against their true masks. Print it.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.eval(finetuned_score, metric="miou_rle", stage="finetuned")


## Task 18: Hand in the fine-tuned file

Write `submission.csv` from the fine-tuned model, in the format of `sample`. Print its shape.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.submit("submission.csv", stage="finetuned")


---
# Part 6 -- Challenge: your best file

Your files are on the board. Now make the model better, one change at a time. A change worth keeping comes from something you found in the data or in the rules, and it raises the score of the validation photos.

## Task 19: Look again

Look again at the data and at your model's mistakes, all of them, for something your files do not use yet. Print what you find.

In [ ]:
# Your code here


## Task 20: Try a better way

Make at least one change to your pipeline. Score it on your validation photos with `score`, and print it next to `finetuned_score`.

In [ ]:
# Your code here


## Task 21: One more try

Try one more change, scored the same way on your validation photos. Keep it only if the score goes up.

In [ ]:
# Your code here


## Task 22: Hand in your best file

Write `submission.csv` again with your best way. Your last accepted file is your answer, so hand in a new file only when you expect it to be better.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.submit("submission.csv", stage="finetuned")


---
## Task 23: Reflect

Which change moved your validation score most, and how did you find it: in the data, in the rules or by trying? What did the frozen stage give, and what did fine-tuning add? If you met this task again with one hour, what would you do in the first 15 minutes?

*Your answers here*

---
# Finish

Run the last cell when you are done. It sends what is still waiting and hands in this notebook.

In [ ]:
ck.finish()

---
## Conclusion

- **Read the task first**: the scorer read as code, two files without a model, what mIoU rewards, and the format of the file, all before any model.
- **A first honest file**: a U-Net with a pretrained encoder, frozen, scored on validation photos with the task's own metric, handed in early.
- **Pretrained, then fine-tuned**: the same model trained in two stages, each scored and handed in.
- **Look before you trust**: the masks counted and the model's mistakes split by class, and what was found used.
- **Keep a change only when the score says so**: each change scored on validation photos with the task's own metric before a new file.
- **Debugging**: a shape error read from the error message, located and repaired against the clock.

---

Made By **Sattam Altwaim**